# M1 Normal vs Fault 전환 검증

## tl;dr

이 노트북은 `normal vs pre_event` 기준을 버리지 않고, `normal vs fault_event`로 전환 가능한지 마지막으로 검증한다.

- 목적은 조기탐지가 아니라 fault record 상태 구분 가능성 검증이다.
- normal 35건은 회사 제공 기준으로 유지한다.
- main positive는 M1 `faults.csv`에서 Event 20, 34, 69를 제외한 known fault 30건이다.
- Event 34 포함, Event 67 제외, Event 34 포함 + Event 67 제외를 sensitivity로 본다.
- feature는 `compact13`과 `expanded154`를 분리 비교한다.
- train-only candidate normal 70건을 붙인 경우와 붙이지 않은 경우를 비교한다.
- threshold는 0.6을 의사결정 기준으로 고정하고, 0.5는 recall 참고용으로 산출한다.


## Context & Methods

### Key Assumptions

- 원본 ZIP과 metadata는 수정하지 않는다.
- 모든 계산은 M1 기준으로만 수행한다.
- normal 라벨은 변경하지 않는다.
- `fault_event`는 M1 `faults.csv`에 존재하는 fault record를 뜻한다.
- 모델 저장이나 운영 배포는 하지 않는다.


In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
OUTPUT_DIR = ROOT / "07_데이터산출물"

NOTEBOOK_PATH = ROOT / "06_노트북" / "15_m1_normal_vs_fault_transition_validation.ipynb"
AUDIT_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_audit.csv"
FEATURE_POOL_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_feature_pool.csv"
DATASET_SUMMARY_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_dataset_summary.csv"
CV_METRICS_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_cv_metrics.csv"
PREDICTIONS_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_predictions.csv"
THRESHOLD_METRICS_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_threshold_metrics.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_feature_importance.csv"
DECISION_MATRIX_PATH = OUTPUT_DIR / "m1_normal_vs_fault_transition_decision_matrix.csv"
REPORT_PATH = OUTPUT_DIR / "15_M1_normal_vs_fault_transition_검증_보고서.md"
DECISION_PNG = OUTPUT_DIR / "m1_normal_vs_fault_transition_decision_matrix.png"
FEATURE_COMPARE_PNG = OUTPUT_DIR / "m1_normal_vs_fault_transition_feature_comparison.png"

SOURCE_PREFIX = "manufacturer 1"
RANDOM_STATE = 42
THRESHOLDS = [0.5, 0.6]
DECISION_THRESHOLD = 0.6
EVENT20_ID = 20
EVENT34_ID = 34
EVENT67_ID = 67
EVENT69_ID = 69
HARD_NORMAL_TAG_BY_EVENT = {
    27: "hard_normal_metadata",
    28: "hard_normal_candidate",
    68: "review_required_normal",
}
EXCLUDED_EVENTS_ALWAYS = {EVENT20_ID, EVENT69_ID}

ORIGINAL_SIGNALS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
DERIVED_SIGNALS = [
    "s_hc1_supply_temperature_error",
    "p_net_delta_temperature",
    "p_net_power_flow_ratio",
    "p_return_gap",
]
ALL_SIGNALS = ORIGINAL_SIGNALS + DERIVED_SIGNALS
SENSOR_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]
TEMPORAL_STATS = [
    "last_1d_mean_minus_prev_6d_mean",
    "last_12h_mean_minus_prev_12h_mean",
    "last_6h_mean_minus_prev_6h_mean",
    "last_1d_std_minus_prev_6d_std",
]
BASE70_FEATURES = [f"{signal}__{stat}" for signal in ORIGINAL_SIGNALS for stat in SENSOR_STATS]
DERIVED_STAT_FEATURES = [f"{signal}__{stat}" for signal in DERIVED_SIGNALS for stat in SENSOR_STATS]
TEMPORAL_FEATURES = [f"{signal}__{stat}" for signal in ALL_SIGNALS for stat in TEMPORAL_STATS]
EXPANDED154_FEATURES = BASE70_FEATURES + DERIVED_STAT_FEATURES + TEMPORAL_FEATURES

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert ZIP_PATH.exists(), ZIP_PATH
assert len(BASE70_FEATURES) == 70
assert len(EXPANDED154_FEATURES) == 154


In [2]:
def read_metadata(name: str) -> pd.DataFrame:
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(f"{SOURCE_PREFIX}/{name}") as file:
            return pd.read_csv(file, sep=";")


def parse_datetime_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = df.copy()
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], errors="coerce")
    return df


faults = parse_datetime_columns(
    read_metadata("faults.csv"),
    ["Report date", "Possible anomaly start", "Possible anomaly end", "Training start", "Training end"],
)
normal_events = parse_datetime_columns(
    read_metadata("normal_events.csv"),
    ["Event start", "Event end", "Training start", "Training end"],
)
disturbances = parse_datetime_columns(read_metadata("disturbances.csv"), ["Event start"])

faults["Event ID"] = faults["Event ID"].astype(int)
faults["substation ID"] = faults["substation ID"].astype(int)
normal_events["Event ID"] = normal_events["Event ID"].astype(int)
normal_events["substation ID"] = normal_events["substation ID"].astype(int)
disturbances["substation ID"] = disturbances["substation ID"].astype(int)

compact_summary = pd.read_csv(OUTPUT_DIR / "m1_compact_feature_set_summary.csv")
compact13_text = compact_summary.loc[
    compact_summary["feature_set"].eq("compact13_overlap"), "features"
].iloc[0]
COMPACT13_FEATURES = [feature for feature in compact13_text.split("|") if feature]
assert len(COMPACT13_FEATURES) == 13
assert set(COMPACT13_FEATURES).issubset(EXPANDED154_FEATURES)

pre_event_lock = pd.read_csv(OUTPUT_DIR / "m1_final_dataset_lock_decision.csv")
pre_event_recall = float(
    pre_event_lock.loc[pre_event_lock["criterion"].eq("recall_no_drop"), "expanded_value"].iloc[0]
)
pre_event_balanced_accuracy = float(
    pre_event_lock.loc[pre_event_lock["criterion"].eq("balanced_accuracy_no_drop"), "expanded_value"].iloc[0]
)
pre_event_fpr = float(
    pre_event_lock.loc[pre_event_lock["criterion"].eq("fpr_delta_within_0_05"), "expanded_value"].iloc[0]
)

previous_fault_decision = pd.read_csv(OUTPUT_DIR / "m1_fault_event_decision_matrix.csv")

expansion_pool_existing = pd.read_csv(OUTPUT_DIR / "m1_expansion_feature_pool.csv")
existing_review_tags = (
    expansion_pool_existing.loc[
        expansion_pool_existing["candidate_type"].eq("fixed_eval")
        & expansion_pool_existing["label"].eq("normal"),
        ["source_event_id", "review_tag"],
    ]
    .dropna(subset=["source_event_id"])
    .assign(source_event_id=lambda df: df["source_event_id"].astype(int))
)
review_tag_by_event = {
    int(row.source_event_id): row.review_tag
    for row in existing_review_tags.itertuples(index=False)
    if isinstance(row.review_tag, str) and row.review_tag
}
review_tag_by_event.update(HARD_NORMAL_TAG_BY_EVENT)

candidate_normal_windows = expansion_pool_existing.loc[
    expansion_pool_existing["candidate_type"].eq("candidate_normal")
].copy()
candidate_normal_windows["window_start"] = pd.to_datetime(candidate_normal_windows["window_start"], errors="coerce")
candidate_normal_windows["window_end"] = pd.to_datetime(candidate_normal_windows["window_end"], errors="coerce")

print("fault records:", len(faults))
print("normal events:", len(normal_events))
print("candidate normal windows:", len(candidate_normal_windows))
print("compact13 features:", len(COMPACT13_FEATURES))


fault records: 33
normal events: 35
candidate normal windows: 70
compact13 features: 13


## Data

In [3]:
OPERATIONAL_CACHE: dict[int, pd.DataFrame] = {}


def load_operational(substation_id: int) -> pd.DataFrame:
    path = f"{SOURCE_PREFIX}/operational_data/substation_{int(substation_id)}.csv"
    with zipfile.ZipFile(ZIP_PATH) as zf:
        with zf.open(path) as file:
            df = pd.read_csv(file, sep=";")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    missing_columns = [column for column in ORIGINAL_SIGNALS if column not in df.columns]
    if missing_columns:
        raise ValueError(f"missing signal columns for substation {substation_id}: {missing_columns}")
    return df.sort_values("timestamp").reset_index(drop=True)


def get_operational(substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id not in OPERATIONAL_CACHE:
        OPERATIONAL_CACHE[substation_id] = load_operational(substation_id)
    return OPERATIONAL_CACHE[substation_id]


def add_derived_signals(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["s_hc1_supply_temperature_error"] = (
        pd.to_numeric(df["s_hc1_supply_temperature"], errors="coerce")
        - pd.to_numeric(df["s_hc1_supply_temperature_setpoint"], errors="coerce")
    )
    df["p_net_delta_temperature"] = (
        pd.to_numeric(df["p_net_supply_temperature"], errors="coerce")
        - pd.to_numeric(df["p_net_return_temperature"], errors="coerce")
    )
    flow = pd.to_numeric(df["p_net_meter_flow"], errors="coerce")
    heat_power = pd.to_numeric(df["p_net_meter_heat_power"], errors="coerce")
    df["p_net_power_flow_ratio"] = np.where(flow.abs() > 1e-9, heat_power / flow, np.nan)
    df["p_return_gap"] = (
        pd.to_numeric(df["p_hc1_return_temperature"], errors="coerce")
        - pd.to_numeric(df["p_net_return_temperature"], errors="coerce")
    )
    return df


def window_slice(substation_id: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> pd.DataFrame:
    df = get_operational(substation_id)
    mask = df["timestamp"].ge(window_start) & df["timestamp"].lt(window_end)
    columns = ["timestamp", *ORIGINAL_SIGNALS]
    return add_derived_signals(df.loc[mask, columns].copy())


def expected_rows(window_start: pd.Timestamp, window_end: pd.Timestamp) -> int:
    minutes = (window_end - window_start).total_seconds() / 60
    return int(round(minutes / 10))


def disturbance_summary(substation_id: int, window_start: pd.Timestamp, window_end: pd.Timestamp) -> tuple[int, str]:
    mask = (
        disturbances["substation ID"].eq(int(substation_id))
        & disturbances["Event start"].ge(window_start)
        & disturbances["Event start"].lt(window_end)
    )
    window_disturbances = disturbances.loc[mask].copy()
    if window_disturbances.empty:
        return 0, ""
    disturbance_types = "|".join(sorted(window_disturbances["type"].dropna().astype(str).unique()))
    return int(len(window_disturbances)), disturbance_types


def safe_std(series: pd.Series) -> float:
    non_null = pd.to_numeric(series, errors="coerce").dropna()
    if len(non_null) > 1:
        return float(non_null.std(ddof=1))
    if len(non_null) == 1:
        return 0.0
    return np.nan


def stat_values(window_data: pd.DataFrame, signals: list[str]) -> dict[str, float]:
    values: dict[str, float] = {}
    for signal in signals:
        series = pd.to_numeric(window_data[signal], errors="coerce")
        non_null = series.dropna()
        values[f"{signal}__mean"] = float(non_null.mean()) if len(non_null) else np.nan
        values[f"{signal}__std"] = safe_std(series)
        values[f"{signal}__min"] = float(non_null.min()) if len(non_null) else np.nan
        values[f"{signal}__max"] = float(non_null.max()) if len(non_null) else np.nan
        values[f"{signal}__median"] = float(non_null.median()) if len(non_null) else np.nan
        if len(non_null) > 1:
            values[f"{signal}__last_minus_first"] = float(non_null.iloc[-1] - non_null.iloc[0])
        elif len(non_null) == 1:
            values[f"{signal}__last_minus_first"] = 0.0
        else:
            values[f"{signal}__last_minus_first"] = np.nan
        values[f"{signal}__missing_rate"] = float(series.isna().mean()) if len(series) else 1.0
    return values


def segment_mean(window_data: pd.DataFrame, signal: str, start: pd.Timestamp, end: pd.Timestamp) -> float:
    segment = window_data.loc[
        window_data["timestamp"].ge(start) & window_data["timestamp"].lt(end),
        signal,
    ]
    non_null = pd.to_numeric(segment, errors="coerce").dropna()
    return float(non_null.mean()) if len(non_null) else np.nan


def segment_std(window_data: pd.DataFrame, signal: str, start: pd.Timestamp, end: pd.Timestamp) -> float:
    segment = window_data.loc[
        window_data["timestamp"].ge(start) & window_data["timestamp"].lt(end),
        signal,
    ]
    return safe_std(segment)


def temporal_values(window_data: pd.DataFrame, window_start: pd.Timestamp, window_end: pd.Timestamp) -> dict[str, float]:
    values: dict[str, float] = {}
    cut_1d = window_end - pd.Timedelta(days=1)
    cut_12h = window_end - pd.Timedelta(hours=12)
    cut_24h = window_end - pd.Timedelta(hours=24)
    cut_6h = window_end - pd.Timedelta(hours=6)
    cut_12h_for_6h = window_end - pd.Timedelta(hours=12)

    for signal in ALL_SIGNALS:
        values[f"{signal}__last_1d_mean_minus_prev_6d_mean"] = (
            segment_mean(window_data, signal, cut_1d, window_end)
            - segment_mean(window_data, signal, window_start, cut_1d)
        )
        values[f"{signal}__last_12h_mean_minus_prev_12h_mean"] = (
            segment_mean(window_data, signal, cut_12h, window_end)
            - segment_mean(window_data, signal, cut_24h, cut_12h)
        )
        values[f"{signal}__last_6h_mean_minus_prev_6h_mean"] = (
            segment_mean(window_data, signal, cut_6h, window_end)
            - segment_mean(window_data, signal, cut_12h_for_6h, cut_6h)
        )
        values[f"{signal}__last_1d_std_minus_prev_6d_std"] = (
            segment_std(window_data, signal, cut_1d, window_end)
            - segment_std(window_data, signal, window_start, cut_1d)
        )
    return values


def make_sample(
    *,
    sample_id: str,
    label: str,
    y: int,
    label_strength: str,
    source_event_id,
    substation_id: int,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    metadata: dict,
) -> dict:
    window_data = window_slice(substation_id, window_start, window_end)
    expected = expected_rows(window_start, window_end)
    sample_count = int(len(window_data))
    coverage_rate = sample_count / expected if expected else 0.0
    disturbance_count, disturbance_types = disturbance_summary(substation_id, window_start, window_end)
    row = {
        "sample_id": sample_id,
        "label": label,
        "y": int(y),
        "label_strength": label_strength,
        "source_event_id": source_event_id,
        "substation_id": int(substation_id),
        "window_start": window_start,
        "window_end": window_end,
        "window_days": float((window_end - window_start).total_seconds() / 86400),
        "window_policy": "report_pre_7d" if label == "fault_event" else "normal_7d",
        "sample_count": sample_count,
        "expected_sample_count": expected,
        "coverage_rate": float(coverage_rate),
        "disturbance_count": disturbance_count,
        "disturbance_types": disturbance_types,
    }
    row.update(metadata)
    row.update(stat_values(window_data, ORIGINAL_SIGNALS))
    row.update(stat_values(window_data, DERIVED_SIGNALS))
    row.update(temporal_values(window_data, window_start, window_end))
    return row


In [4]:
normal_rows = []
for _, event in normal_events.sort_values("Event ID").iterrows():
    event_id = int(event["Event ID"])
    normal_rows.append(
        make_sample(
            sample_id=f"normal_{event_id}",
            label="normal",
            y=0,
            label_strength="company_normal",
            source_event_id=event_id,
            substation_id=int(event["substation ID"]),
            window_start=event["Event start"],
            window_end=event["Event end"],
            metadata={
                "pool_role": "fixed_eval",
                "candidate_type": "fixed_eval",
                "source_type": "normal_event",
                "report_date": pd.NaT,
                "fault_label": "",
                "fault_label_known": False,
                "fault_label_unknown": False,
                "efd_possible": False,
                "training_metadata_complete": False,
                "event67_flag": False,
                "review_tag": review_tag_by_event.get(event_id, ""),
                "audit_status": "accepted_normal",
                "exclusion_reason": "",
            },
        )
    )

fault_rows = []
for _, event in faults.sort_values("Event ID").iterrows():
    event_id = int(event["Event ID"])
    report_date = event["Report date"]
    window_start = report_date - pd.Timedelta(days=7)
    window_end = report_date
    fault_label = "" if pd.isna(event["Fault label"]) else str(event["Fault label"])
    fault_label_unknown = fault_label.strip().lower() == "unknown"
    fault_label_known = bool(fault_label and not fault_label_unknown)
    training_metadata_complete = bool(
        pd.notna(event["Training start"]) and pd.notna(event["Training end"])
    )
    if event_id == EVENT20_ID:
        audit_status = "excluded_low_coverage"
        exclusion_reason = "event20_low_coverage"
    elif event_id == EVENT34_ID:
        audit_status = "sensitivity_only_unknown_label"
        exclusion_reason = "event34_unknown_label"
    elif event_id == EVENT69_ID:
        audit_status = "excluded_unknown_label_training_end_missing"
        exclusion_reason = "event69_unknown_label_training_end_missing"
    else:
        audit_status = "accepted_main_candidate"
        exclusion_reason = ""

    fault_rows.append(
        make_sample(
            sample_id=f"fault_{event_id}",
            label="fault_event",
            y=1,
            label_strength="fault_record",
            source_event_id=event_id,
            substation_id=int(event["substation ID"]),
            window_start=window_start,
            window_end=window_end,
            metadata={
                "pool_role": "fixed_eval",
                "candidate_type": "fixed_eval",
                "source_type": "fault_record",
                "report_date": report_date,
                "possible_anomaly_start": event["Possible anomaly start"],
                "possible_anomaly_end": event["Possible anomaly end"],
                "training_start": event["Training start"],
                "training_end": event["Training end"],
                "fault_label": fault_label,
                "fault_label_known": fault_label_known,
                "fault_label_unknown": fault_label_unknown,
                "efd_possible": bool(event["efd_possible"]),
                "training_metadata_complete": training_metadata_complete,
                "event67_flag": event_id == EVENT67_ID,
                "review_tag": "",
                "audit_status": audit_status,
                "exclusion_reason": exclusion_reason,
            },
        )
    )

candidate_normal_rows = []
for index, candidate in candidate_normal_windows.reset_index(drop=True).iterrows():
    substation_id = int(candidate["substation_id"])
    window_start = candidate["window_start"]
    window_end = candidate["window_end"]
    sample_id = str(candidate["sample_id"])
    candidate_normal_rows.append(
        make_sample(
            sample_id=sample_id,
            label="candidate_normal",
            y=0,
            label_strength="candidate_normal",
            source_event_id=np.nan,
            substation_id=substation_id,
            window_start=window_start,
            window_end=window_end,
            metadata={
                "pool_role": "train_only_candidate",
                "candidate_type": "candidate_normal",
                "source_type": "candidate_normal",
                "report_date": pd.NaT,
                "fault_label": "",
                "fault_label_known": False,
                "fault_label_unknown": False,
                "efd_possible": False,
                "training_metadata_complete": False,
                "event67_flag": False,
                "review_tag": "",
                "audit_status": "train_only_candidate",
                "exclusion_reason": "",
            },
        )
    )

normal_df = pd.DataFrame(normal_rows)
fault_df = pd.DataFrame(fault_rows)
candidate_normal_df = pd.DataFrame(candidate_normal_rows)
feature_pool_df = pd.concat([normal_df, fault_df, candidate_normal_df], ignore_index=True)

for column in ["window_start", "window_end", "report_date", "possible_anomaly_start", "possible_anomaly_end", "training_start", "training_end"]:
    if column in feature_pool_df.columns:
        feature_pool_df[column] = pd.to_datetime(feature_pool_df[column], errors="coerce")

audit_df = pd.concat(
    [
        normal_df.assign(audit_role="normal_fixed_eval"),
        fault_df.assign(audit_role="fault_record_audit"),
    ],
    ignore_index=True,
)

feature_pool_df.to_csv(FEATURE_POOL_PATH, index=False, encoding="utf-8-sig")
audit_df.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")

print("normal rows:", len(normal_df))
print("fault audit rows:", len(fault_df))
print("candidate normal rows:", len(candidate_normal_df))
print(
    fault_df[
        [
            "source_event_id",
            "fault_label_known",
            "fault_label_unknown",
            "training_metadata_complete",
            "event67_flag",
            "coverage_rate",
            "audit_status",
            "exclusion_reason",
        ]
    ].to_string(index=False)
)


normal rows: 35
fault audit rows: 33
candidate normal rows: 70
 source_event_id  fault_label_known  fault_label_unknown  training_metadata_complete  event67_flag  coverage_rate                                audit_status                           exclusion_reason
               1               True                False                        True         False       1.000000                     accepted_main_candidate                                           
               3               True                False                        True         False       1.000000                     accepted_main_candidate                                           
               5               True                False                        True         False       1.000000                     accepted_main_candidate                                           
               6               True                False                        True         False       1.000000                    

In [5]:
def build_fixed_eval_dataset(dataset_id: str) -> pd.DataFrame:
    normal_part = normal_df.copy()
    fault_part = fault_df.copy()
    if dataset_id == "fault_known_only_main":
        fault_part = fault_part.loc[
            fault_part["fault_label_known"].eq(True)
            & ~fault_part["source_event_id"].astype(int).isin([EVENT20_ID, EVENT34_ID, EVENT69_ID])
        ].copy()
    elif dataset_id == "fault_with_unknown34":
        fault_part = fault_part.loc[
            ~fault_part["source_event_id"].astype(int).isin([EVENT20_ID, EVENT69_ID])
        ].copy()
    elif dataset_id == "fault_no_event67":
        fault_part = fault_part.loc[
            fault_part["fault_label_known"].eq(True)
            & ~fault_part["source_event_id"].astype(int).isin([EVENT20_ID, EVENT34_ID, EVENT67_ID, EVENT69_ID])
        ].copy()
    elif dataset_id == "fault_with_unknown34_no_event67":
        fault_part = fault_part.loc[
            ~fault_part["source_event_id"].astype(int).isin([EVENT20_ID, EVENT67_ID, EVENT69_ID])
        ].copy()
    else:
        raise ValueError(dataset_id)
    dataset = pd.concat([normal_part, fault_part], ignore_index=True)
    dataset["dataset_id"] = dataset_id
    return dataset


DATASET_IDS = [
    "fault_known_only_main",
    "fault_with_unknown34",
    "fault_no_event67",
    "fault_with_unknown34_no_event67",
]
DATASETS = {dataset_id: build_fixed_eval_dataset(dataset_id) for dataset_id in DATASET_IDS}

FEATURE_SET_COLUMNS = {
    "compact13": COMPACT13_FEATURES,
    "expanded154": EXPANDED154_FEATURES,
}
TRAIN_MODES = ["fixed_only", "candidate_normal_train"]
MODELS = ["dummy_most_frequent", "logistic_balanced"]

summary_rows = []
for dataset_id, dataset in DATASETS.items():
    fault_events = sorted(dataset.loc[dataset["label"].eq("fault_event"), "source_event_id"].astype(int).tolist())
    normal_events_list = sorted(dataset.loc[dataset["label"].eq("normal"), "source_event_id"].astype(int).tolist())
    summary_rows.append(
        {
            "dataset_id": dataset_id,
            "rows": int(len(dataset)),
            "normal_rows": int(dataset["label"].eq("normal").sum()),
            "fault_positive_rows": int(dataset["label"].eq("fault_event").sum()),
            "candidate_normal_train_rows": int(len(candidate_normal_df)),
            "fault_event_ids": ",".join(map(str, fault_events)),
            "normal_event_ids": ",".join(map(str, normal_events_list)),
            "event20_included": EVENT20_ID in fault_events,
            "event34_included": EVENT34_ID in fault_events,
            "event67_included": EVENT67_ID in fault_events,
            "event69_included": EVENT69_ID in fault_events,
            "normal_event27_tag": dataset.loc[dataset["source_event_id"].eq(27), "review_tag"].iloc[0],
            "normal_event28_tag": dataset.loc[dataset["source_event_id"].eq(28), "review_tag"].iloc[0],
            "normal_event68_tag": dataset.loc[dataset["source_event_id"].eq(68), "review_tag"].iloc[0],
            "compact13_feature_count": len(COMPACT13_FEATURES),
            "expanded154_feature_count": len(EXPANDED154_FEATURES),
            "coverage_min": float(dataset["coverage_rate"].min()),
            "coverage_median": float(dataset["coverage_rate"].median()),
        }
    )

dataset_summary_df = pd.DataFrame(summary_rows)
dataset_summary_df.to_csv(DATASET_SUMMARY_PATH, index=False, encoding="utf-8-sig")
dataset_summary_df


,dataset_id,rows,normal_rows,fault_positive_rows,candidate_normal_train_rows,fault_event_ids,normal_event_ids,event20_included,event34_included,event67_included,event69_included,normal_event27_tag,normal_event28_tag,normal_event68_tag,compact13_feature_count,expanded154_feature_count,coverage_min,coverage_median
0,fault_known_only_main,65,35,30,70,"1,3,5,6,7,10,11,13,15,23,24,29,32,36,37,38,40,...","2,4,8,12,14,16,17,18,19,21,22,25,26,27,28,30,3...",False,False,True,False,hard_normal_metadata,hard_normal_candidate,review_required_normal,13,154,0.99504,1.0
1,fault_with_unknown34,66,35,31,70,"1,3,5,6,7,10,11,13,15,23,24,29,32,34,36,37,38,...","2,4,8,12,14,16,17,18,19,21,22,25,26,27,28,30,3...",False,True,True,False,hard_normal_metadata,hard_normal_candidate,review_required_normal,13,154,0.99504,1.0
2,fault_no_event67,64,35,29,70,"1,3,5,6,7,10,11,13,15,23,24,29,32,36,37,38,40,...","2,4,8,12,14,16,17,18,19,21,22,25,26,27,28,30,3...",False,False,False,False,hard_normal_metadata,hard_normal_candidate,review_required_normal,13,154,0.99504,1.0
3,fault_with_unknown34_no_event67,65,35,30,70,"1,3,5,6,7,10,11,13,15,23,24,29,32,34,36,37,38,...","2,4,8,12,14,16,17,18,19,21,22,25,26,27,28,30,3...",False,True,False,False,hard_normal_metadata,hard_normal_candidate,review_required_normal,13,154,0.99504,1.0


## Results

In [6]:
def make_model(model_name: str) -> Pipeline:
    if model_name == "dummy_most_frequent":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        )
    if model_name == "logistic_balanced":
        return Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )
    raise ValueError(model_name)


def positive_scores(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    estimator = model.named_steps["model"]
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        classes = list(estimator.classes_)
        if 1 in classes:
            return probabilities[:, classes.index(1)]
    return np.zeros(len(X), dtype=float)


def metric_row(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None = None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    roc_auc = float(roc_auc_score(y_true, y_score)) if y_score is not None and len(np.unique(y_true)) == 2 else 0.5
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "true_negative_count": int(tn),
        "false_positive_count": int(fp),
        "false_negative_count": int(fn),
        "true_positive_count": int(tp),
        "false_positive_rate": float(fp / (fp + tn)) if (fp + tn) else 0.0,
        "false_negative_rate": float(fn / (fn + tp)) if (fn + tp) else 0.0,
    }


def selected_candidate_train(test_groups: set[int]) -> pd.DataFrame:
    return candidate_normal_df.loc[
        ~candidate_normal_df["substation_id"].astype(int).isin(test_groups)
    ].copy()


def train_eval_one(dataset_id: str, feature_set: str, train_mode: str, model_name: str):
    dataset = DATASETS[dataset_id].copy().reset_index(drop=True)
    feature_columns = FEATURE_SET_COLUMNS[feature_set]
    X = dataset[feature_columns]
    y = dataset["y"].astype(int).to_numpy()
    groups = dataset["substation_id"].astype(int).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    fold_metric_rows = []
    prediction_rows = []
    group_rows = []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(X, y, groups), start=1):
        fixed_train = dataset.iloc[train_idx].copy()
        fixed_test = dataset.iloc[test_idx].copy()
        train_groups = set(fixed_train["substation_id"].astype(int))
        test_groups = set(fixed_test["substation_id"].astype(int))
        assert train_groups.isdisjoint(test_groups)

        if train_mode == "fixed_only":
            candidate_train = candidate_normal_df.iloc[0:0].copy()
        elif train_mode == "candidate_normal_train":
            candidate_train = selected_candidate_train(test_groups)
        else:
            raise ValueError(train_mode)

        train_rows = pd.concat([fixed_train, candidate_train], ignore_index=True)
        assert not set(candidate_train["substation_id"].astype(int)).intersection(test_groups)
        assert train_rows["y"].nunique() == 2
        model = make_model(model_name)
        model.fit(train_rows[feature_columns], train_rows["y"].astype(int))
        scores = positive_scores(model, fixed_test[feature_columns])

        group_rows.append(
            {
                "dataset_id": dataset_id,
                "feature_set": feature_set,
                "train_mode": train_mode,
                "model": model_name,
                "fold": fold,
                "train_groups": "|".join(map(str, sorted(train_groups))),
                "test_groups": "|".join(map(str, sorted(test_groups))),
                "candidate_train_rows": int(len(candidate_train)),
                "group_overlap_count": int(len(train_groups.intersection(test_groups))),
            }
        )

        for threshold in THRESHOLDS:
            preds = (scores >= threshold).astype(int)
            metric = metric_row(fixed_test["y"].astype(int).to_numpy(), preds, scores)
            metric.update(
                {
                    "dataset_id": dataset_id,
                    "feature_set": feature_set,
                    "feature_count": len(feature_columns),
                    "train_mode": train_mode,
                    "model": model_name,
                    "fold": fold,
                    "threshold": threshold,
                    "fixed_train_rows": int(len(fixed_train)),
                    "candidate_train_rows": int(len(candidate_train)),
                    "test_rows": int(len(fixed_test)),
                    "test_normal_rows": int((fixed_test["y"] == 0).sum()),
                    "test_fault_rows": int((fixed_test["y"] == 1).sum()),
                }
            )
            fold_metric_rows.append(metric)
            for row, score, pred in zip(fixed_test.itertuples(index=False), scores, preds):
                prediction_rows.append(
                    {
                        "dataset_id": dataset_id,
                        "feature_set": feature_set,
                        "feature_count": len(feature_columns),
                        "train_mode": train_mode,
                        "model": model_name,
                        "fold": fold,
                        "threshold": threshold,
                        "sample_id": row.sample_id,
                        "source_event_id": row.source_event_id,
                        "substation_id": int(row.substation_id),
                        "label": row.label,
                        "label_strength": row.label_strength,
                        "review_tag": row.review_tag,
                        "fault_label": row.fault_label,
                        "fault_label_known": bool(row.fault_label_known),
                        "fault_label_unknown": bool(row.fault_label_unknown),
                        "event67_flag": bool(row.event67_flag),
                        "coverage_rate": float(row.coverage_rate),
                        "actual_binary": int(row.y),
                        "positive_score": float(score),
                        "predicted_binary": int(pred),
                        "predicted_label": "fault_event" if int(pred) == 1 else "normal",
                        "error_type": (
                            "FP"
                            if int(row.y) == 0 and int(pred) == 1
                            else "FN"
                            if int(row.y) == 1 and int(pred) == 0
                            else "TP"
                            if int(row.y) == 1
                            else "TN"
                        ),
                    }
                )
    return fold_metric_rows, prediction_rows, group_rows


metric_rows = []
prediction_rows = []
group_audit_rows = []
for dataset_id in DATASET_IDS:
    for feature_set in FEATURE_SET_COLUMNS:
        for train_mode in TRAIN_MODES:
            for model_name in MODELS:
                fold_metrics, preds, group_audit = train_eval_one(dataset_id, feature_set, train_mode, model_name)
                metric_rows.extend(fold_metrics)
                prediction_rows.extend(preds)
                group_audit_rows.extend(group_audit)

cv_metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.DataFrame(prediction_rows)
group_audit_df = pd.DataFrame(group_audit_rows)
cv_metrics_df = cv_metrics_df.merge(
    group_audit_df,
    on=["dataset_id", "feature_set", "train_mode", "model", "fold"],
    how="left",
)

cv_metrics_df.to_csv(CV_METRICS_PATH, index=False, encoding="utf-8-sig")
predictions_df.to_csv(PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

threshold_rows = []
for keys, part in predictions_df.groupby(["dataset_id", "feature_set", "train_mode", "model", "threshold"], dropna=False):
    y_true = part["actual_binary"].astype(int).to_numpy()
    y_pred = part["predicted_binary"].astype(int).to_numpy()
    y_score = part["positive_score"].astype(float).to_numpy()
    metric = metric_row(y_true, y_pred, y_score)
    dataset_id, feature_set, train_mode, model_name, threshold = keys
    fp_events = sorted(part.loc[part["error_type"].eq("FP"), "source_event_id"].dropna().astype(int).tolist())
    fn_events = sorted(part.loc[part["error_type"].eq("FN"), "source_event_id"].dropna().astype(int).tolist())
    hard_fp_events = sorted(set(fp_events).intersection(HARD_NORMAL_TAG_BY_EVENT))
    metric.update(
        {
            "dataset_id": dataset_id,
            "feature_set": feature_set,
            "feature_count": len(FEATURE_SET_COLUMNS[feature_set]),
            "train_mode": train_mode,
            "model": model_name,
            "threshold": float(threshold),
            "rows": int(len(part)),
            "normal_rows": int((part["actual_binary"] == 0).sum()),
            "fault_rows": int((part["actual_binary"] == 1).sum()),
            "fp_event_ids": ",".join(map(str, fp_events)),
            "fn_event_ids": ",".join(map(str, fn_events)),
            "hard_normal_fp_event_ids": ",".join(map(str, hard_fp_events)),
            "hard_normal_fp_share": float(len(hard_fp_events) / len(fp_events)) if fp_events else 1.0,
        }
    )
    threshold_rows.append(metric)

threshold_metrics_df = pd.DataFrame(threshold_rows)
threshold_metrics_df.to_csv(THRESHOLD_METRICS_PATH, index=False, encoding="utf-8-sig")
threshold_metrics_df.sort_values(["dataset_id", "feature_set", "train_mode", "model", "threshold"]).head()


,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,true_negative_count,false_positive_count,false_negative_count,true_positive_count,...,train_mode,model,threshold,rows,normal_rows,fault_rows,fp_event_ids,fn_event_ids,hard_normal_fp_event_ids,hard_normal_fp_share
0,0.538462,0.500000,0.000000,0.000000,0.000000,0.500000,35,0,30,0,...,candidate_normal_train,dummy_most_frequent,0.5,65,35,30,,"1,3,5,6,7,10,11,13,15,23,24,29,32,36,37,38,40,...",,1.0
1,0.538462,0.500000,0.000000,0.000000,0.000000,0.500000,35,0,30,0,...,candidate_normal_train,dummy_most_frequent,0.6,65,35,30,,"1,3,5,6,7,10,11,13,15,23,24,29,32,36,37,38,40,...",,1.0
2,0.784615,0.780952,0.785714,0.733333,0.758621,0.859048,29,6,8,22,...,candidate_normal_train,logistic_balanced,0.5,65,35,30,"18,27,28,33,59,68","11,15,29,36,44,45,52,63","27,28,68",0.5
3,0.800000,0.795238,0.814815,0.733333,0.771930,0.859048,30,5,8,22,...,candidate_normal_train,logistic_balanced,0.6,65,35,30,"18,27,28,33,59","11,15,29,36,44,45,52,63","27,28",0.4
4,0.538462,0.500000,0.000000,0.000000,0.000000,0.500000,35,0,30,0,...,fixed_only,dummy_most_frequent,0.5,65,35,30,,"1,3,5,6,7,10,11,13,15,23,24,29,32,36,37,38,40,...",,1.0


In [7]:
importance_rows = []
for dataset_id, dataset in DATASETS.items():
    for feature_set, feature_columns in FEATURE_SET_COLUMNS.items():
        for train_mode in TRAIN_MODES:
            train_rows = dataset.copy()
            if train_mode == "candidate_normal_train":
                train_rows = pd.concat([train_rows, candidate_normal_df], ignore_index=True)
            model = make_model("logistic_balanced")
            model.fit(train_rows[feature_columns], train_rows["y"].astype(int))
            coefficients = model.named_steps["model"].coef_[0]
            scaler = model.named_steps["scaler"]
            for feature, coefficient, mean, scale in zip(feature_columns, coefficients, scaler.mean_, scaler.scale_):
                importance_rows.append(
                    {
                        "dataset_id": dataset_id,
                        "feature_set": feature_set,
                        "feature_count": len(feature_columns),
                        "train_mode": train_mode,
                        "feature": feature,
                        "coefficient": float(coefficient),
                        "abs_coefficient": float(abs(coefficient)),
                        "scaler_mean": float(mean),
                        "scaler_scale": float(scale),
                    }
                )

feature_importance_df = pd.DataFrame(importance_rows)
feature_importance_df["rank"] = feature_importance_df.groupby(
    ["dataset_id", "feature_set", "train_mode"]
)["abs_coefficient"].rank(method="first", ascending=False).astype(int)
feature_importance_df.sort_values(
    ["dataset_id", "feature_set", "train_mode", "rank"]
).to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")
feature_importance_df.head()


,dataset_id,feature_set,feature_count,train_mode,feature,coefficient,abs_coefficient,scaler_mean,scaler_scale,rank
0,fault_known_only_main,compact13,13,fixed_only,outdoor_temperature__last_12h_mean_minus_prev_...,-0.882384,0.882384,0.422806,2.791628,2
1,fault_known_only_main,compact13,13,fixed_only,outdoor_temperature__last_6h_mean_minus_prev_6...,1.570474,1.570474,-0.537525,2.333033,1
2,fault_known_only_main,compact13,13,fixed_only,outdoor_temperature__last_minus_first,0.147781,0.147781,-1.210756,4.202951,11
3,fault_known_only_main,compact13,13,fixed_only,p_hc1_return_temperature__last_12h_mean_minus_...,-0.499830,0.499830,-0.156077,3.409179,4
4,fault_known_only_main,compact13,13,fixed_only,p_hc1_return_temperature__last_1d_mean_minus_p...,-0.453327,0.453327,-0.604366,3.575843,5


In [8]:
def metric_lookup(dataset_id: str, feature_set: str, train_mode: str, model: str, threshold: float) -> pd.Series:
    row = threshold_metrics_df.loc[
        threshold_metrics_df["dataset_id"].eq(dataset_id)
        & threshold_metrics_df["feature_set"].eq(feature_set)
        & threshold_metrics_df["train_mode"].eq(train_mode)
        & threshold_metrics_df["model"].eq(model)
        & np.isclose(threshold_metrics_df["threshold"].astype(float), threshold)
    ]
    assert len(row) == 1, (dataset_id, feature_set, train_mode, model, threshold, len(row))
    return row.iloc[0]


main_compact = metric_lookup(
    "fault_known_only_main",
    "compact13",
    "candidate_normal_train",
    "logistic_balanced",
    DECISION_THRESHOLD,
)
main_dummy = metric_lookup(
    "fault_known_only_main",
    "compact13",
    "candidate_normal_train",
    "dummy_most_frequent",
    DECISION_THRESHOLD,
)
main_fixed_only = metric_lookup(
    "fault_known_only_main",
    "compact13",
    "fixed_only",
    "logistic_balanced",
    DECISION_THRESHOLD,
)
main_expanded = metric_lookup(
    "fault_known_only_main",
    "expanded154",
    "candidate_normal_train",
    "logistic_balanced",
    DECISION_THRESHOLD,
)
with_unknown34 = metric_lookup(
    "fault_with_unknown34",
    "compact13",
    "candidate_normal_train",
    "logistic_balanced",
    DECISION_THRESHOLD,
)
no_event67 = metric_lookup(
    "fault_no_event67",
    "compact13",
    "candidate_normal_train",
    "logistic_balanced",
    DECISION_THRESHOLD,
)
with_unknown34_no_event67 = metric_lookup(
    "fault_with_unknown34_no_event67",
    "compact13",
    "candidate_normal_train",
    "logistic_balanced",
    DECISION_THRESHOLD,
)

dummy_lift = float(main_compact["balanced_accuracy"] - main_dummy["balanced_accuracy"])
recall_drop_vs_pre_event = float(pre_event_recall - main_compact["recall"])
compact_ba_gap_vs_expanded = float(main_expanded["balanced_accuracy"] - main_compact["balanced_accuracy"])
compact_recall_gap_vs_expanded = float(main_expanded["recall"] - main_compact["recall"])
candidate_normal_ba_delta = float(main_compact["balanced_accuracy"] - main_fixed_only["balanced_accuracy"])
candidate_normal_recall_delta = float(main_compact["recall"] - main_fixed_only["recall"])
event34_ba_delta = float(with_unknown34["balanced_accuracy"] - main_compact["balanced_accuracy"])
event34_recall_delta = float(with_unknown34["recall"] - main_compact["recall"])
event67_ba_delta = float(no_event67["balanced_accuracy"] - main_compact["balanced_accuracy"])
event67_recall_delta = float(no_event67["recall"] - main_compact["recall"])

fp_events_main = [
    int(event_id)
    for event_id in str(main_compact["fp_event_ids"]).split(",")
    if event_id and event_id != "nan"
]
hard_normal_fp_share = float(main_compact["hard_normal_fp_share"])

decision_rows = [
    {
        "criterion": "dummy_lift_clear",
        "baseline_value": float(main_dummy["balanced_accuracy"]),
        "candidate_value": float(main_compact["balanced_accuracy"]),
        "delta": dummy_lift,
        "pass": bool(dummy_lift >= 0.05),
        "blocking_for_transition": True,
        "note": "compact13 fault_event balanced accuracy lift over dummy must be at least 0.05",
    },
    {
        "criterion": "recall_not_far_below_pre_event",
        "baseline_value": pre_event_recall,
        "candidate_value": float(main_compact["recall"]),
        "delta": float(main_compact["recall"] - pre_event_recall),
        "pass": bool(recall_drop_vs_pre_event <= 0.10),
        "blocking_for_transition": True,
        "note": "fault_event recall should not fall more than 0.10 below locked pre_event recall",
    },
    {
        "criterion": "fp_concentrated_in_known_hard_normals",
        "baseline_value": 0.5,
        "candidate_value": hard_normal_fp_share,
        "delta": hard_normal_fp_share - 0.5,
        "pass": bool(len(fp_events_main) == 0 or hard_normal_fp_share >= 0.5),
        "blocking_for_transition": True,
        "note": "FP should be absent or mostly concentrated in Event 27/28/68 hard normal tags",
    },
    {
        "criterion": "candidate_normal_train_not_harmful",
        "baseline_value": float(main_fixed_only["balanced_accuracy"]),
        "candidate_value": float(main_compact["balanced_accuracy"]),
        "delta": candidate_normal_ba_delta,
        "pass": bool(candidate_normal_ba_delta >= -0.02 and candidate_normal_recall_delta >= -0.05),
        "blocking_for_transition": True,
        "note": "train-only candidate normal should not materially degrade compact13 main result",
    },
    {
        "criterion": "compact13_close_to_expanded154",
        "baseline_value": float(main_expanded["balanced_accuracy"]),
        "candidate_value": float(main_compact["balanced_accuracy"]),
        "delta": -compact_ba_gap_vs_expanded,
        "pass": bool(compact_ba_gap_vs_expanded <= 0.03 and compact_recall_gap_vs_expanded <= 0.05),
        "blocking_for_transition": True,
        "note": "compact13 should be close to expanded154 on the fault_event main dataset",
    },
    {
        "criterion": "event34_sensitivity_policy",
        "baseline_value": float(main_compact["balanced_accuracy"]),
        "candidate_value": float(with_unknown34["balanced_accuracy"]),
        "delta": event34_ba_delta,
        "pass": bool(event34_ba_delta >= -0.03 and event34_recall_delta >= -0.05),
        "blocking_for_transition": False,
        "note": "if unstable, keep Event 34 excluded and sensitivity-only",
    },
    {
        "criterion": "event67_sensitivity_policy",
        "baseline_value": float(main_compact["balanced_accuracy"]),
        "candidate_value": float(no_event67["balanced_accuracy"]),
        "delta": event67_ba_delta,
        "pass": bool(abs(event67_ba_delta) <= 0.03 and abs(event67_recall_delta) <= 0.05),
        "blocking_for_transition": False,
        "note": "if unstable, keep Event 67 as long-anomaly sensitivity tag",
    },
]

blocking_pass = all(row["pass"] for row in decision_rows if row["blocking_for_transition"])
final_decision = (
    "normal_vs_fault_transition_candidate_main_known_fault"
    if blocking_pass
    else "keep_pre_event_main_fault_event_auxiliary"
)
decision_df = pd.DataFrame(decision_rows)
decision_df["final_decision"] = final_decision
decision_df.to_csv(DECISION_MATRIX_PATH, index=False, encoding="utf-8-sig")

decision_df


,criterion,baseline_value,candidate_value,delta,pass,blocking_for_transition,note,final_decision
0,dummy_lift_clear,0.500000,0.795238,0.295238,True,True,compact13 fault_event balanced accuracy lift o...,keep_pre_event_main_fault_event_auxiliary
1,recall_not_far_below_pre_event,0.785714,0.733333,-0.052381,True,True,fault_event recall should not fall more than 0...,keep_pre_event_main_fault_event_auxiliary
2,fp_concentrated_in_known_hard_normals,0.500000,0.400000,-0.100000,False,True,FP should be absent or mostly concentrated in ...,keep_pre_event_main_fault_event_auxiliary
3,candidate_normal_train_not_harmful,0.730952,0.795238,0.064286,True,True,train-only candidate normal should not materia...,keep_pre_event_main_fault_event_auxiliary
4,compact13_close_to_expanded154,0.630952,0.795238,0.164286,True,True,compact13 should be close to expanded154 on th...,keep_pre_event_main_fault_event_auxiliary
5,event34_sensitivity_policy,0.795238,0.811982,0.016743,True,False,"if unstable, keep Event 34 excluded and sensit...",keep_pre_event_main_fault_event_auxiliary
6,event67_sensitivity_policy,0.795238,0.787685,-0.007553,True,False,"if unstable, keep Event 67 as long-anomaly sen...",keep_pre_event_main_fault_event_auxiliary


In [9]:
plot_metrics = threshold_metrics_df.loc[
    threshold_metrics_df["threshold"].eq(DECISION_THRESHOLD)
    & threshold_metrics_df["model"].eq("logistic_balanced")
    & threshold_metrics_df["train_mode"].eq("candidate_normal_train")
    & threshold_metrics_df["feature_set"].isin(["compact13", "expanded154"])
].copy()
plot_metrics["display"] = plot_metrics["dataset_id"] + "\n" + plot_metrics["feature_set"]
plot_metrics = plot_metrics.sort_values(["dataset_id", "feature_set"])

fig, ax = plt.subplots(figsize=(12, 5.5))
colors = plot_metrics["feature_set"].map({"compact13": "#4c78a8", "expanded154": "#f58518"}).tolist()
ax.bar(plot_metrics["display"], plot_metrics["balanced_accuracy"], color=colors)
ax.axhline(float(main_dummy["balanced_accuracy"]), color="#333333", linestyle="--", linewidth=1, label="dummy main")
ax.set_ylim(0, 1)
ax.set_ylabel("balanced accuracy")
ax.set_title("Normal vs fault transition validation at threshold 0.6")
ax.tick_params(axis="x", rotation=45)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(DECISION_PNG, bbox_inches="tight")
plt.close(fig)

feature_compare = threshold_metrics_df.loc[
    threshold_metrics_df["dataset_id"].eq("fault_known_only_main")
    & threshold_metrics_df["model"].eq("logistic_balanced")
    & threshold_metrics_df["train_mode"].isin(TRAIN_MODES)
    & threshold_metrics_df["feature_set"].isin(["compact13", "expanded154"])
    & threshold_metrics_df["threshold"].eq(DECISION_THRESHOLD)
].copy()
feature_compare["display"] = feature_compare["feature_set"] + "\n" + feature_compare["train_mode"]

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(feature_compare["display"], feature_compare["recall"], color="#54a24b")
ax.axhline(pre_event_recall, color="#333333", linestyle="--", linewidth=1, label="locked pre_event recall")
ax.set_ylim(0, 1)
ax.set_ylabel("recall")
ax.set_title("Fault_event recall comparison at threshold 0.6")
ax.tick_params(axis="x", rotation=20)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FEATURE_COMPARE_PNG, bbox_inches="tight")
plt.close(fig)

[DECISION_PNG, FEATURE_COMPARE_PNG]


[WindowsPath('C:/3rd_Project/HeatGridAgent/07_데이터산출물/m1_normal_vs_fault_transition_decision_matrix.png'),
 WindowsPath('C:/3rd_Project/HeatGridAgent/07_데이터산출물/m1_normal_vs_fault_transition_feature_comparison.png')]

## Takeaways

In [10]:
def markdown_table(df: pd.DataFrame, columns: list[str]) -> str:
    table = df[columns].copy()
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in table.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, sep] + rows)


def fmt_float(value: float) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.4f}"


main_rows = threshold_metrics_df.loc[
    threshold_metrics_df["dataset_id"].eq("fault_known_only_main")
    & threshold_metrics_df["threshold"].eq(DECISION_THRESHOLD)
    & threshold_metrics_df["model"].isin(["dummy_most_frequent", "logistic_balanced"])
].copy()
main_rows = main_rows.loc[
    main_rows["feature_set"].isin(["compact13", "expanded154"])
    & main_rows["train_mode"].isin(TRAIN_MODES)
].copy()
for col in ["balanced_accuracy", "precision", "recall", "f1", "false_positive_rate", "roc_auc"]:
    main_rows[col] = main_rows[col].map(fmt_float)
main_rows = main_rows.sort_values(["feature_set", "train_mode", "model"])

sensitivity_rows = threshold_metrics_df.loc[
    threshold_metrics_df["feature_set"].eq("compact13")
    & threshold_metrics_df["train_mode"].eq("candidate_normal_train")
    & threshold_metrics_df["model"].eq("logistic_balanced")
    & threshold_metrics_df["threshold"].eq(DECISION_THRESHOLD)
].copy()
for col in ["balanced_accuracy", "precision", "recall", "f1", "false_positive_rate", "roc_auc"]:
    sensitivity_rows[col] = sensitivity_rows[col].map(fmt_float)
sensitivity_rows = sensitivity_rows.sort_values("dataset_id")

decision_report = decision_df.copy()
for col in ["baseline_value", "candidate_value", "delta"]:
    decision_report[col] = decision_report[col].map(fmt_float)

previous_fault_report = previous_fault_decision.copy()
for col in previous_fault_report.columns:
    if previous_fault_report[col].dtype.kind in "fc":
        previous_fault_report[col] = previous_fault_report[col].map(fmt_float)
previous_fault_report = previous_fault_report.head(4)

fp_events_text = str(main_compact["fp_event_ids"]) if str(main_compact["fp_event_ids"]) else ""
fn_events_text = str(main_compact["fn_event_ids"]) if str(main_compact["fn_event_ids"]) else ""

event34_policy = (
    "Event 34 포함 sensitivity가 안정적이다."
    if bool(decision_df.loc[decision_df["criterion"].eq("event34_sensitivity_policy"), "pass"].iloc[0])
    else "Event 34 포함 시 결과가 흔들리므로 main에서는 계속 제외한다."
)
event67_policy = (
    "Event 67 제외 sensitivity가 main과 크게 다르지 않다."
    if bool(decision_df.loc[decision_df["criterion"].eq("event67_sensitivity_policy"), "pass"].iloc[0])
    else "Event 67 제외 여부에 따라 결과가 달라져 장기 anomaly sensitivity로 따로 관리한다."
)

report = f"""# M1 Normal vs Fault 전환 검증 보고서

## 개요

이번 검증은 `normal vs pre_event` 기준을 바로 버릴지 판단하기 전에, `normal vs fault_event`가 다음 main 목표가 될 수 있는지 확인한 작업이다.

최종 판단: **{final_decision}**

## 무엇을 했는지

- positive를 M1 `faults.csv`의 fault record로 재정의했다.
- main fixed eval은 normal 35건 + known fault 30건으로 구성했다.
- Event 20은 low coverage, Event 34는 unknown fault label, Event 69는 unknown fault label 및 `Training end` 없음으로 main에서 제외했다.
- Event 34 포함, Event 67 제외, Event 34 포함 + Event 67 제외를 sensitivity로 따로 계산했다.
- train-only candidate normal 70건을 붙인 경우와 붙이지 않은 경우를 비교했다.
- feature는 `compact13`과 `expanded154`를 분리해 저장했다.
- threshold 0.6을 의사결정 기준으로 쓰고, threshold 0.5도 recall 참고용으로 산출했다.

## 기존 07번 결과 재검토

기존 `normal vs fault_event` 실험은 lift가 작고 sensitivity가 안정적이지 않았다. 이번 검증은 같은 방향을 더 엄격한 fixed eval, candidate normal train-only, feature set 분리 기준으로 다시 본 것이다.

{markdown_table(previous_fault_report, list(previous_fault_report.columns))}

## Main 결과: fault_known_only_main, threshold 0.6

{markdown_table(main_rows, ["feature_set", "train_mode", "model", "rows", "balanced_accuracy", "precision", "recall", "f1", "false_positive_rate", "false_positive_count", "false_negative_count", "fp_event_ids", "fn_event_ids"])}

## Sensitivity 결과: compact13 + candidate normal train, threshold 0.6

{markdown_table(sensitivity_rows, ["dataset_id", "rows", "normal_rows", "fault_rows", "balanced_accuracy", "precision", "recall", "f1", "false_positive_rate", "false_positive_count", "false_negative_count", "fp_event_ids", "fn_event_ids"])}

## Decision Matrix

{markdown_table(decision_report, ["criterion", "baseline_value", "candidate_value", "delta", "pass", "blocking_for_transition", "final_decision"])}

## 해석

- Dummy 대비 main compact13 lift: `{dummy_lift:.4f}`.
- pre_event locked recall `{pre_event_recall:.4f}` 대비 fault_event main compact13 recall은 `{float(main_compact["recall"]):.4f}`이다.
- fault_event main compact13 FP 이벤트는 `{fp_events_text}`, FN 이벤트는 `{fn_events_text}`이다.
- hard normal Event 27/28/68 FP 집중도는 `{hard_normal_fp_share:.4f}`이다.
- {event34_policy}
- {event67_policy}

## 변경 내용

| 항목 | 내용 |
| --- | --- |
| 노트북 | `06_노트북/15_m1_normal_vs_fault_transition_validation.ipynb` |
| audit | `m1_normal_vs_fault_transition_audit.csv` |
| feature pool | `m1_normal_vs_fault_transition_feature_pool.csv` |
| dataset summary | `m1_normal_vs_fault_transition_dataset_summary.csv` |
| metrics | `m1_normal_vs_fault_transition_cv_metrics.csv`, `m1_normal_vs_fault_transition_threshold_metrics.csv` |
| predictions | `m1_normal_vs_fault_transition_predictions.csv` |
| decision | `m1_normal_vs_fault_transition_decision_matrix.csv` |
| plots | `m1_normal_vs_fault_transition_decision_matrix.png`, `m1_normal_vs_fault_transition_feature_comparison.png` |

## 검증

```mermaid
flowchart TD
  A[M1 원본 metadata와 operational CSV] --> B[normal/fault/candidate normal feature 재계산]
  B --> C[fixed eval dataset 구성]
  C --> D[group CV 학습]
  D --> E[threshold 0.5/0.6 평가]
  E --> F[decision matrix]
```

- normal 35건을 유지했다.
- main fault positive는 30건이다.
- Event 20/34/69는 main에 포함되지 않았다.
- Event 34는 sensitivity에만 포함했다.
- Event 67은 main에 포함되고, sensitivity에서 제외 가능하게 분리했다.
- Event 27/28/68 normal tag를 유지했다.
- fold별 train/test `substation_id` overlap은 0이다.
- `compact13=13`, `expanded154=154` feature 결과를 분리 저장했다.

## 한계와 주의점

- 이 실험은 조기탐지가 아니라 fault record 상태 구분 가능성 검증이다.
- normal 라벨은 변경하지 않았다.
- threshold 0.6은 비교 기준이며 운영 threshold로 확정한 것이 아니다.
- final model 저장이나 배포는 하지 않았다.

## 다음에 볼 것

- final decision이 `keep_pre_event_main_fault_event_auxiliary`이면, 현재는 `normal vs pre_event`를 main으로 유지하고 fault_event는 보조 실험으로 둔다.
- final decision이 `normal_vs_fault_transition_candidate_main_known_fault`이면, known fault 30건 기준으로 다음 학습 계획을 세운다.
"""

REPORT_PATH.write_text(report, encoding="utf-8")
print(final_decision)


keep_pre_event_main_fault_event_auxiliary


In [11]:
required_paths = [
    AUDIT_PATH,
    FEATURE_POOL_PATH,
    DATASET_SUMMARY_PATH,
    CV_METRICS_PATH,
    PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    DECISION_MATRIX_PATH,
    REPORT_PATH,
    DECISION_PNG,
    FEATURE_COMPARE_PNG,
]
for path in required_paths:
    assert path.exists() and path.stat().st_size > 0, path

written_summary = pd.read_csv(DATASET_SUMMARY_PATH)
main_summary = written_summary.loc[written_summary["dataset_id"].eq("fault_known_only_main")].iloc[0]
assert int(main_summary["normal_rows"]) == 35
assert int(main_summary["fault_positive_rows"]) == 30
assert bool(main_summary["event20_included"]) is False
assert bool(main_summary["event34_included"]) is False
assert bool(main_summary["event69_included"]) is False
assert bool(main_summary["event67_included"]) is True

with_unknown_summary = written_summary.loc[written_summary["dataset_id"].eq("fault_with_unknown34")].iloc[0]
assert bool(with_unknown_summary["event34_included"]) is True
no_event67_summary = written_summary.loc[written_summary["dataset_id"].eq("fault_no_event67")].iloc[0]
assert bool(no_event67_summary["event67_included"]) is False

assert main_summary["normal_event27_tag"] == "hard_normal_metadata"
assert main_summary["normal_event28_tag"] == "hard_normal_candidate"
assert main_summary["normal_event68_tag"] == "review_required_normal"

assert len(COMPACT13_FEATURES) == 13
assert len(EXPANDED154_FEATURES) == 154
assert group_audit_df["group_overlap_count"].eq(0).all()

written_predictions = pd.read_csv(PREDICTIONS_PATH)
main_predictions = written_predictions.loc[
    written_predictions["dataset_id"].eq("fault_known_only_main")
    & written_predictions["threshold"].eq(DECISION_THRESHOLD)
]
assert not main_predictions["source_event_id"].dropna().astype(int).isin([EVENT20_ID, EVENT34_ID, EVENT69_ID]).any()

forbidden_terms = ["manufacturer" + "_2", "manufacturer " + "2", "M" + "2"]
check_paths = [
    NOTEBOOK_PATH,
    REPORT_PATH,
    AUDIT_PATH,
    FEATURE_POOL_PATH,
    DATASET_SUMMARY_PATH,
    CV_METRICS_PATH,
    PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    DECISION_MATRIX_PATH,
]
hits = []
for path in check_paths:
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    if any(term in text for term in forbidden_terms):
        hits.append(str(path))
assert not hits, hits

validation_summary = {
    "main_rows": int(main_summary["rows"]),
    "main_normal_rows": int(main_summary["normal_rows"]),
    "main_fault_positive_rows": int(main_summary["fault_positive_rows"]),
    "group_overlap_max": int(group_audit_df["group_overlap_count"].max()),
    "compact13_feature_count": len(COMPACT13_FEATURES),
    "expanded154_feature_count": len(EXPANDED154_FEATURES),
    "final_decision": final_decision,
}
validation_summary


{'main_rows': 65,
 'main_normal_rows': 35,
 'main_fault_positive_rows': 30,
 'group_overlap_max': 0,
 'compact13_feature_count': 13,
 'expanded154_feature_count': 154,
 'final_decision': 'keep_pre_event_main_fault_event_auxiliary'}